# **Artificial Neural Networks and Deep Learning ~ Challenge 1**

---

## **Improved Model: CNN + BiLSTM + Attention + Focal Loss + Enhanced Features**

## ⚙️ **Libraries Import and Initial Configuration**

In [ ]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os
import warnings

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
from typing import Dict, Optional, Callable
from collections import defaultdict
import time
import datetime

# Import Numpy and Pandas
import numpy as np
import pandas as pd

# Import sklearn and imblearn
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE

import torch
torch.manual_seed(SEED)
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, Subset
import torch.nn.functional as F

# Set seeds for random number generators
np.random.seed(SEED)
random.seed(SEED)

# Platform-specific device configuration
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## ⏳ **Data Loading**

In [ ]:
# Define the PATH to your notebook
PATH_TO_YOUR_DATASET="./data/"
dataframe_train_raw = pd.read_csv(PATH_TO_YOUR_DATASET+'pirate_pain_train.csv')
dataframe_train_labels = pd.read_csv(PATH_TO_YOUR_DATASET+'pirate_pain_train_labels.csv')
dataframe_test_raw = pd.read_csv(PATH_TO_YOUR_DATASET+'pirate_pain_test.csv')

## 🔎 **Data Cleaning and Feature Engineering**

In [ ]:
# Merge train data with labels
dataframe = dataframe_train_raw.merge(dataframe_train_labels, on='sample_index', how='left')

# ADVICE 15/11: Data Cleaning - Remove columns with issues
# Drop columns with excessive missing values or low variance
dataframe = dataframe.drop(['joint_30', 'joint_24', 'joint_13', 'n_hands', 'n_eyes'], axis=1)

# Convert n_legs to binary patient_class (pirate=1 leg, ordinary=2 legs)
dataframe['n_legs'] = dataframe['n_legs'].apply(lambda x: 2 if x == 'two' else 1)
dataframe.rename(columns={'n_legs': 'patient_class'}, inplace=True)

print(f"DataFrame shape: {dataframe.shape}")

# ADVICE 12/11: Time Feature Engineering - Add cyclic time features
# Normalize time to [0, 1] range per subject, then create cyclic features
for sid in dataframe['sample_index'].unique():
    mask = dataframe['sample_index'] == sid
    time_vals = dataframe.loc[mask, 'time'].values
    if len(time_vals) > 1:
        # Normalize to [0, 2*pi] for cyclic encoding
        time_normalized = (time_vals - time_vals.min()) / (time_vals.max() - time_vals.min() + 1e-8)
        dataframe.loc[mask, 'time_sin'] = np.sin(2 * np.pi * time_normalized)
        dataframe.loc[mask, 'time_cos'] = np.cos(2 * np.pi * time_normalized)
    else:
        dataframe.loc[mask, 'time_sin'] = 0.0
        dataframe.loc[mask, 'time_cos'] = 0.0

# Convert to float32 for efficiency
for col in dataframe.columns:
    if col.startswith('joint_') or col.startswith('pain_survey_') or col == 'patient_class' or 'time' in col:
        dataframe[col] = dataframe[col].astype(np.float32)

LABELS_MAPPING = {
    'no_pain': 0,
    'low_pain': 1,
    'high_pain': 2,
}

print("\nFeature engineering complete.")
print(f"Columns: {list(dataframe.columns)}")

## 📊 **ADVICE 11/11: Autocorrelation Analysis for Window Size**

In [ ]:
# Analyze autocorrelation to determine optimal window size
# We'll sample a few subjects to compute average autocorrelation

def compute_autocorr(series, max_lag=100):
    """
    Compute autocorrelation for a 1D series up to max_lag
    :param series: 1D numpy array of time series data
    :param max_lag: maximum lag to compute
    :return: autocorrelation values for lags 1 to max_lag
    """
    n = len(series)
    if n < max_lag:
        max_lag = n - 1

    mean = np.mean(series)
    var = np.var(series)
    if var == 0:
        return np.zeros(max_lag)

    autocorr = np.zeros(max_lag)
    for lag in range(1, max_lag):
        c0 = np.sum((series[:-lag] - mean) * (series[lag:] - mean)) / n
        autocorr[lag] = c0 / var
    return autocorr

# Sample subjects and compute average autocorrelation on joint features
sample_subjects = np.random.choice(dataframe['sample_index'].unique(), size=min(20, len(dataframe['sample_index'].unique())), replace=False)
joint_cols = [col for col in dataframe.columns if col.startswith('joint_')]

autocorr_curves = []
for sid in sample_subjects:
    subject_data = dataframe[dataframe['sample_index'] == sid][joint_cols].values
    if len(subject_data) > 50:
        # Average across all joint features
        avg_series = subject_data.mean(axis=1)
        ac = compute_autocorr(avg_series, max_lag=min(100, len(avg_series)))
        autocorr_curves.append(ac)

if autocorr_curves:
    # Average autocorrelation across subjects
    min_len = min(len(ac) for ac in autocorr_curves)
    avg_autocorr = np.mean([ac[:min_len] for ac in autocorr_curves], axis=0)

    # Find where autocorrelation drops below 0.1 (weak correlation)
    optimal_window = np.argmax(avg_autocorr < 0.1)
    if optimal_window == 0:
        optimal_window = 30  # fallback

    print(f"\nAutocorrelation analysis suggests optimal window size: {optimal_window}")
    print(f"We'll use WINDOW_SIZE = {optimal_window}")
    WINDOW_SIZE = optimal_window
else:
    WINDOW_SIZE = 30
    print(f"\nUsing default WINDOW_SIZE = {WINDOW_SIZE}")

# Use window size from autocorrelation analysis
STRIDE = max(1, WINDOW_SIZE // 5)  # 20% overlap
print(f"STRIDE = {STRIDE}")

## 🔄 **Data Preprocessing & Stratified Split (SUBJECT-LEVEL)**

In [ ]:
# Subject-level stratified split - FIX FOR DATA LEAKAGE
subject_df = (
    dataframe[['sample_index', 'label']]
    .drop_duplicates(subset='sample_index')
    .sort_values('sample_index')
)

unique_ids = subject_df['sample_index'].values
subject_labels = subject_df['label'].values

# Stratified split at subject level (75% train, 25% test)
trainval_ids, test_ids = train_test_split(
    unique_ids,
    test_size=0.15,
    random_state=SEED,
    shuffle=True,
    stratify=subject_labels,
)

df_train = dataframe[dataframe['sample_index'].isin(trainval_ids)].copy()
df_test = dataframe[dataframe['sample_index'].isin(test_ids)].copy()

# Map labels to integers
label_mapping = {'no_pain': 0, 'low_pain': 1, 'high_pain': 2}
df_train['label'] = df_train['label'].map(label_mapping)
df_test['label'] = df_test['label'].map(label_mapping)

print(f"Train subjects: {len(trainval_ids)}, Test subjects: {len(test_ids)}")
print(f"Train samples: {len(df_train)}, Test samples: {len(df_test)}")
print(f"Train label distribution: {df_train.groupby('label')['sample_index'].nunique().to_dict()}")
print(f"Test label distribution: {df_test.groupby('label')['sample_index'].nunique().to_dict()}")

## 🔢 **Data Normalization**

In [ ]:
# Define columns for normalization
joint_cols = [col for col in df_train.columns if col.startswith('joint_')]
survey_cols = [col for col in df_train.columns if col.startswith('pain_survey_')]
time_cols = ['time', 'time_sin', 'time_cos']

col_to_normalize = joint_cols + time_cols
scale_columns = col_to_normalize + survey_cols  # all features for sequences

# Calculate min/max from training data only
mins = df_train[col_to_normalize].min()
maxs = df_train[col_to_normalize].max()

# Normalize train and test
for column in col_to_normalize:
    range_val = maxs[column] - mins[column]
    if range_val == 0:
        range_val = 1.0
    df_train[column] = (df_train[column] - mins[column]) / range_val
    df_test[column] = (df_test[column] - mins[column]) / range_val

# Process actual test set (for submission)
df_submission = dataframe_test_raw.copy()
df_submission = df_submission.drop(['joint_30', 'joint_24', 'joint_13', 'n_hands', 'n_eyes'], axis=1)
df_submission['n_legs'] = df_submission['n_legs'].apply(lambda x: 2 if x == 'two' else 1)
df_submission.rename(columns={'n_legs': 'patient_class'}, inplace=True)

# Add time features to submission set
for sid in df_submission['sample_index'].unique():
    mask = df_submission['sample_index'] == sid
    time_vals = df_submission.loc[mask, 'time'].values
    if len(time_vals) > 1:
        time_normalized = (time_vals - time_vals.min()) / (time_vals.max() - time_vals.min() + 1e-8)
        df_submission.loc[mask, 'time_sin'] = np.sin(2 * np.pi * time_normalized)
        df_submission.loc[mask, 'time_cos'] = np.cos(2 * np.pi * time_normalized)
    else:
        df_submission.loc[mask, 'time_sin'] = 0.0
        df_submission.loc[mask, 'time_cos'] = 0.0

# Normalize submission set using train statistics
for column in col_to_normalize:
    if column in df_submission.columns:
        range_val = maxs[column] - mins[column]
        if range_val == 0:
            range_val = 1.0
        df_submission[column] = (df_submission[column] - mins[column]) / range_val

print("Normalization complete.")
print(f"Features to use: {len(scale_columns)} total")

## 🪟 **Windowing with Sequences**

In [ ]:
PATIENT_CLASS_DIM = 1
PATIENT_CLASS_EMB_DIM = 16  # Increased embedding dimension for patient class

def build_sequences(df, window=30, stride=6):
    """ Build overlapping sequences from dataframe
    :param df: input dataframe
    :param window: size of each window
    :param stride: step size between windows
    :return: dataset, labels, sample_ids, patient_features arrays
     """
    dataset = []
    labels = []
    sids = []
    patient_feats = []

    # Process each subject separately
    for sid in df['sample_index'].unique():
        temp = df[df['sample_index'] == sid][scale_columns].values
        label = df[df['sample_index'] == sid]['label'].values[0] if 'label' in df.columns else -1
        p_feat = df[df['sample_index'] == sid]['patient_class'].values[0]

        # Adding padding if needed
        if len(temp) < window:
            padding_len = window - len(temp)
            padding = np.zeros((padding_len, len(scale_columns)), dtype=np.float32)
            temp = np.concatenate((temp, padding))

        # Creation of overlapping windows
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            labels.append(label)
            sids.append(sid)
            patient_feats.append(p_feat)
            idx += stride

    dataset = np.array(dataset, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)
    sids = np.array(sids, dtype=np.int64)
    patient_feats = np.array(patient_feats, dtype=np.float32).reshape(-1, 1)

    return dataset, labels, sids, patient_feats

# Build sequences for train, test, and submission sets
X_train, y_train, sid_train, pfeat_train = build_sequences(df_train, WINDOW_SIZE, STRIDE)
X_test, y_test, sid_test, pfeat_test = build_sequences(df_test, WINDOW_SIZE, STRIDE)
X_submission, y_submission, sid_submission, pfeat_submission = build_sequences(df_submission, WINDOW_SIZE, STRIDE)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"X_submission shape: {X_submission.shape}")
print(f"y_train distribution: {np.bincount(y_train)}")
print(f"y_test distribution: {np.bincount(y_test)}")

## 🏗️ **Model Architecture: CNN + BiLSTM + Attention**

### ADVICE 13/11: 1D Convolutions for local pattern extraction
### ADVICE 14/11: Attention mechanism for important timesteps

In [1]:
class CNNBiLSTMAttention(nn.Module):
    """
    Enhanced classifier with:
    - 1D CNN front-end for local pattern extraction (ADVICE 13/11)
    - BiLSTM for temporal modeling
    - Multi-head attention mechanism (ADVICE 14/11)
    - Patient class embedding (ADVICE 07/11)
    - Dropout and layer normalization for regularization
    :param input_size: number of input features
    :param hidden_size: LSTM hidden size
    :param num_layers: number of LSTM layers
    :param num_classes: number of output classes
    :param dropout_rate: dropout rate
    :param patient_emb_dim: dimension of patient class embedding
    :param num_patient_classes: number of patient classes
    :param cnn_channels: tuple of CNN channel sizes
    :param cnn_kernel_size: kernel size for CNN layers
    :param num_attention_heads: number of attention heads
    """
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 128,
        num_layers: int = 2,
        num_classes: int = 3,
        dropout_rate: float = 0.5,
        patient_emb_dim: int = 16,
        num_patient_classes: int = 3,
        cnn_channels: tuple = (64, 128),
        cnn_kernel_size: int = 5,
        num_attention_heads: int = 4,
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # ADVICE 13/11: 1D CNN front-end for local temporal patterns
        c1, c2 = cnn_channels
        padding = cnn_kernel_size // 2

        # Convolutional layers with BatchNorm and ReLU activation and Dropout
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_size, out_channels=c1,
                     kernel_size=cnn_kernel_size, padding=padding),
            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Conv1d(in_channels=c1, out_channels=c2,
                     kernel_size=cnn_kernel_size, padding=padding),
            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
        )
        cnn_out_dim = c2

        # BiLSTM for temporal dependencies
        self.lstm = nn.LSTM(
            input_size=cnn_out_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
        )

        lstm_out_dim = hidden_size * 2  # bidirectional

        # ADVICE 14/11: Multi-head attention mechanism
        self.attention = nn.MultiheadAttention(
            embed_dim=lstm_out_dim,
            num_heads=num_attention_heads,
            dropout=dropout_rate,
            batch_first=True,
        )

        # Layer normalization
        self.layer_norm = nn.LayerNorm(lstm_out_dim)

        # ADVICE 07/11: Patient class embedding
        self.patient_emb = nn.Embedding(
            num_embeddings=num_patient_classes,
            embedding_dim=patient_emb_dim,
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim + patient_emb_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_classes),
        )

    def forward(self, x_seq, p_feat):
        """
        Forward pass of the model
        :param x_seq: (B, T, F)
        :param p_feat: (B, 1) patient class
        """
        B, T, F = x_seq.shape

        # CNN: (B, T, F) -> (B, F, T) -> (B, C, T) -> (B, T, C)
        x_cnn = x_seq.transpose(1, 2)
        x_cnn = self.cnn(x_cnn)
        x_lstm_in = x_cnn.transpose(1, 2)

        # LSTM
        lstm_out, _ = self.lstm(x_lstm_in)  # (B, T, lstm_out_dim)

        # ADVICE 14/11: Self-attention over time
        # Use LSTM output as query, key, value
        attn_out, attn_weights = self.attention(
            query=lstm_out,
            key=lstm_out,
            value=lstm_out,
            need_weights=True,
        )  # (B, T, lstm_out_dim)

        # Add & Norm (residual connection)
        attn_out = self.layer_norm(attn_out + lstm_out)

        # Global average pooling over time
        pooled = attn_out.mean(dim=1)  # (B, lstm_out_dim)

        # Patient embedding
        if p_feat is None:
            p_idx = torch.zeros(B, dtype=torch.long, device=pooled.device)
        else:
            if p_feat.dim() == 2 and p_feat.size(1) == 1:
                p_idx = p_feat.squeeze(1).long()
            else:
                p_idx = p_feat.long()

        # Convert to valid indices (1->0, 2->1)
        p_idx = (p_idx - 1).clamp(0, 1)

        p_emb = self.patient_emb(p_idx)  # (B, patient_emb_dim)

        # Concatenate and classify
        joint_rep = torch.cat([pooled, p_emb], dim=1)
        logits = self.classifier(joint_rep)

        return logits

IndentationError: unexpected indent (2208838281.py, line 10)

## 🎯 **ADVICE 08/11 & 09/11: Focal Loss with Label Smoothing for Class Imbalance**

In [ ]:
class FocalLossWithLabelSmoothing(nn.Module):
    """
    Combines Focal Loss (ADVICE 08/11) with Label Smoothing (ADVICE 09/11)
    to handle class imbalance and prevent overconfidence
    :param gamma: focusing parameter for focal loss
    :param alpha: class weights tensor of shape (num_classes,)
    :param smoothing: label smoothing factor
    :param reduction: reduction method ('mean', 'sum', or 'none')
    """
    def __init__(self, gamma=2.0, alpha=None, smoothing=0.1, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # class weights
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, logits, targets):
        """
        Compute focal loss with label smoothing
        :param logits: (predictions) of shape (B, C)
        :param targets: (true labels) of shape (B),
        :return: loss value
        """
        num_classes = logits.size(-1)

        # ADVICE 09/11: Label smoothing
        # Convert hard labels to soft labels
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        # Compute log probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)

        # Focal loss component: (1 - p_t)^gamma
        # Get probability of true class
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** self.gamma

        # Cross-entropy with label smoothing
        ce_loss = -(true_dist * log_probs).sum(dim=-1)

        # Apply focal weight
        focal_loss = focal_weight * ce_loss

        # ADVICE 08/11: Apply class weights if provided
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_loss = alpha_t * focal_loss

        # Reduction based on specified method
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss

## 🔄 **Training and Validation Functions**

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device, clip_grad_norm=1.0):
    """
    Train for one epoch with gradient clipping (ADVICE 10/11)
    :param model: The neural network model instance
    :param train_loader: DataLoader for training data
    :param criterion: Loss function
    :param optimizer: Optimizer object
    :param scheduler: Learning rate scheduler
    :param device: Computation device (CPU/GPU)
    :param clip_grad_norm: Maximum norm for gradient clipping
    :return: epoch_loss, epoch_f1 metrics
    """
    model.train()

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    for batch_idx, batch in enumerate(train_loader):
        if len(batch) == 4:
            inputs, targets, _, pfeat = batch
        else:
            inputs, targets = batch[:2]
            pfeat = None

        inputs, targets = inputs.to(device), targets.to(device)
        if pfeat is not None:
            pfeat = pfeat.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits = model(inputs, pfeat)
        loss = criterion(logits, targets)

        # Backward pass
        loss.backward()

        # ADVICE 10/11: Gradient clipping to prevent exploding gradients
        if clip_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

        optimizer.step()

        # Metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Step scheduler if provided
    if scheduler is not None:
        scheduler.step()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='macro'  # macro F1 for imbalanced classes
    )

    return epoch_loss, epoch_f1


@torch.no_grad()
def validate_one_epoch(
    model,
    val_loader,
    criterion,
    device,
    int_to_label: Dict[int, str],
    num_classes: int,
    window_aggregation: str = "mean_probs",
    return_predictions: bool = False,
):
    """
    Validate for one epoch with subject-level aggregation
    :param model: The neural network model instance
    :param val_loader: DataLoader for validation data
    :param criterion: Loss function
    :param device: Computation device (CPU/GPU)
    :param int_to_label: Mapping from integer labels to string labels
    :param num_classes: Number of classes
    :param window_aggregation: Method for aggregating window predictions ('mean_probs' or 'vote')
    :param return_predictions: Whether to return final predictions and subject IDs
    :return: epoch_loss, macro_f1, eval_dict, (optional) final_preds, subject_ids
    """
    model.eval()

    running_loss = 0.0
    window_recs = []

    for batch in val_loader:
        if len(batch) == 4:
            inputs, targets, sids, pfeat = batch
        else:
            raise ValueError(f"Expected 4-tuple batch, got {len(batch)}")

        inputs, pfeat = inputs.to(device), pfeat.to(device)
        
        # Handle case where targets might be -1 (submission data)
        if targets[0].item() != -1:
            targets = targets.to(device)
            logits = model(inputs, pfeat)
            loss = criterion(logits, targets)
            running_loss += loss.item() * inputs.size(0)
            labs = targets.cpu().numpy()
        else:
            logits = model(inputs, pfeat)
            labs = np.full(len(inputs), -1)  # placeholder

        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        sids_np = sids.cpu().numpy() if isinstance(sids, torch.Tensor) else np.array(sids)

        for sid, p, lab in zip(sids_np, probs, labs):
            window_recs.append({
                'sid': sid,
                'probs': p,
                'label': lab,
            })

    epoch_loss = running_loss / len(val_loader.dataset) if running_loss > 0 else 0.0

    # Subject-level aggregation
    by_sid = {}
    for r in window_recs:
        by_sid.setdefault(r['sid'], []).append(r)

    final_preds = []
    final_labels = []
    subject_ids = []

    for sid, items in by_sid.items():
        labs = [it['label'] for it in items]
        final_labels.append(labs[0])
        subject_ids.append(sid)

        if window_aggregation == 'mean_probs':
            mean_probs = np.mean(np.stack([it['probs'] for it in items], axis=0), axis=0)
            final_preds.append(int(mean_probs.argmax()))
        elif window_aggregation == 'vote':
            votes = [np.argmax(it['probs']) for it in items]
            counts = np.bincount(votes, minlength=num_classes)
            final_preds.append(int(counts.argmax()))
        else:
            raise ValueError(f"Unknown aggregation: {window_aggregation}")

    final_preds = np.array(final_preds)
    final_labels = np.array(final_labels)
    subject_ids = np.array(subject_ids)

    # Only compute metrics if we have real labels
    if final_labels[0] != -1:
        macro_f1 = f1_score(final_labels, final_preds, average='macro')
        present_labels = sorted(set(final_labels))
        target_names = [int_to_label[i] for i in present_labels]
        report = classification_report(
            final_labels,
            final_preds,
            target_names=target_names,
            digits=4,
            zero_division=0,
        )
        cm = confusion_matrix(final_labels, final_preds, labels=present_labels)
    else:
        macro_f1 = 0.0
        report = "No labels available (submission data)"
        cm = None

    eval_dict = {
        'macro_f1': macro_f1,
        'report': report,
        'confusion_matrix': cm,
        'labels': final_labels,
        'preds': final_preds,
        'subject_ids': subject_ids,
    }

    if return_predictions:
        return epoch_loss, macro_f1, eval_dict, final_preds, subject_ids
    return epoch_loss, macro_f1, eval_dict

## 🔀 **K-Fold Cross-Validation with SMOTE (SUBJECT-LEVEL SPLIT - FIX)**

In [ ]:
class ArrayDataset(Dataset):
    """
    Simple Dataset wrapping NumPy arrays
    :param x_array: input features array
    :param y_array: labels array
    :param sid_array: subject IDs array
    :param p_array: patient features array
    """
    def __init__(self, x_array, y_array, sid_array, p_array):
        self.x = x_array.astype(np.float32)
        self.y = y_array.astype(np.int64)
        self.sids = sid_array.astype(np.int64)
        self.p = p_array.astype(np.float32) if p_array is not None else None

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.x[idx])
        y = torch.tensor(self.y[idx], dtype=torch.long)
        sid = int(self.sids[idx])
        p = None if self.p is None else torch.from_numpy(self.p[idx])
        return x, y, sid, p


def apply_smote_to_fold(X_fold, y_fold, sid_fold, pfeat_fold, target_counts, random_state=42):
    """
    Apply SMOTE oversampling to a fold at the window level,
    then map synthetic samples back to nearest real samples for sid and pfeat
    :param X_fold: input features for the fold (N, T, F)
    :param y_fold: labels for the fold (N,)
    :param sid_fold: subject IDs for the fold (N,)
    :param pfeat_fold: patient features for the fold (N, P) or None
    :param target_counts: dict mapping class label to target count after SMOTE
    :param random_state: random seed for SMOTE
    :return: X_res, y_res, sid_res, pfeat_res after SMOTE
    """
    N, T, F = X_fold.shape
    X_flat = X_fold.reshape(N, T * F)

    # Compute sampling strategy - FIXED for NumPy 2.0+
    unique_classes, class_counts = np.unique(y_fold, return_counts=True)
    counts = dict(zip(unique_classes, class_counts))

    sampling_strategy = {}
    for cls, target in target_counts.items():
        current_count = counts.get(cls, 0)
        # Ensure target is not None
        if target is not None and current_count > 0 and target > current_count:
            sampling_strategy[cls] = min(target, current_count * 5)  # cap at 5x original

    # Determine safe k_neighbors - work with dictionary values
    non_zero_counts = [count for count in counts.values() if count > 0]
    min_class_count = min(non_zero_counts) if non_zero_counts else 1
    k_neighbors = max(1, min(5, min_class_count - 1))

    print(f"  Applying SMOTE: strategy={sampling_strategy}, k_neighbors={k_neighbors}")

    # Define SMOTE instance from imblearn
    sm = SMOTE(
        sampling_strategy=sampling_strategy,
        k_neighbors=k_neighbors,
        random_state=random_state,
    )

    X_res_flat, y_res = sm.fit_resample(X_flat, y_fold)

    # Map synthetic samples to nearest neighbor for sid and pfeat
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(X_flat)
    neigh_idx = nn.kneighbors(X_res_flat, return_distance=False).squeeze(1)

    sid_res = sid_fold[neigh_idx]
    pfeat_res = pfeat_fold[neigh_idx] if pfeat_fold is not None else None

    X_res = X_res_flat.reshape(-1, T, F)

    # Use NumPy 2.0+ compatible counting
    unique_res, counts_res = np.unique(y_res, return_counts=True)
    print(f"  After SMOTE: {dict(zip(unique_res, counts_res))}")

    return X_res, y_res, sid_res, pfeat_res

## ⚙️ **Hyperparameters Configuration**

In [ ]:
# Cross-validation
K_FOLDS = 5 # Number of folds for cross-validation

# Training
EPOCHS = 100 # Number of epochs to train
PATIENCE = 12  # Early stopping patience in epochs
VERBOSE = 1   # Verbosity level

# Optimization
LEARNING_RATE = 1e-3 # Initial learning rate
BATCH_SIZE = 128 # Batch size

# Architecture
HIDDEN_SIZE = 256 # LSTM hidden size
NUM_LAYERS = 2 # Number of LSTM layers
DROPOUT_RATE = 0.4 # Dropout rate
CNN_CHANNELS = (64, 128) # CNN channel sizes
NUM_ATTENTION_HEADS = 8 # Number of attention heads

# Loss function
FOCAL_GAMMA = 2.5      # Focal loss gamma
LABEL_SMOOTHING = 0.15  # ADVICE 09/11

# SMOTE oversampling targets (ADVICE 08/11)
# Target counts per class for more balanced training
SMOTE_TARGET_COUNTS = {
    0: None,  # majority class - don't oversample
    1: 4000,  # low_pain - boost significantly
    2: 3500,  # high_pain - boost significantly
}

print("Hyperparameters set.")
print(f"Window size: {WINDOW_SIZE}, Stride: {STRIDE}")
print(f"Input features: {len(scale_columns)}")

## 🚀 **Stratified K-Fold Training Loop with SMOTE (SUBJECT-LEVEL)**

In [ ]:
# Model saving directory and timestamp for uniqueness
now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
os.makedirs('models', exist_ok=True)

# Prepare dataset
INPUT_FEATURES = X_train.shape[2]
NUM_CLASSES = 3

# Mapping from int to label for reporting
int_to_label = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}

print(f"\n{'='*80}")
print(f"Starting {K_FOLDS}-Fold Cross-Validation (SUBJECT-LEVEL SPLIT)")
print(f"{'='*80}\n")

# Store results
fold_results = []

# Use StratifiedGroupKFold to split by SUBJECTS, not windows
# Get unique subject IDs and their labels from training set
train_subjects = pd.DataFrame({
    'sid': sid_train,
    'label': y_train
}).drop_duplicates(subset='sid')

subject_ids = train_subjects['sid'].values
subject_labels = train_subjects['label'].values

# StratifiedGroupKFold to ensure subject-level splits with stratification by label maintaining class distribution
sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# K-Fold Cross-Validation loop
for fold, (train_subject_idx, val_subject_idx) in enumerate(sgkf.split(X=np.zeros(len(subject_ids)), y=subject_labels, groups=subject_ids), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")

    fold_start_time = time.time()

    # Get subject IDs for this fold
    train_sids = set(subject_ids[train_subject_idx])
    val_sids = set(subject_ids[val_subject_idx])

    print(f"Train subjects: {len(train_sids)}, Val subjects: {len(val_sids)}")

    # Split windows based on subject assignment
    train_mask = np.isin(sid_train, list(train_sids))
    val_mask = np.isin(sid_train, list(val_sids))

    X_tr, X_val = X_train[train_mask], X_train[val_mask]
    y_tr, y_val = y_train[train_mask], y_train[val_mask]
    sid_tr, sid_val = sid_train[train_mask], sid_train[val_mask]
    pfeat_tr, pfeat_val = pfeat_train[train_mask], pfeat_train[val_mask]

    print(f"Train windows: {len(X_tr)}, Val windows: {len(X_val)}")
    unique_tr, counts_tr = np.unique(y_tr, return_counts=True)
    unique_val, counts_val = np.unique(y_val, return_counts=True)
    print(f"Train distribution: {dict(zip(unique_tr, counts_tr))}")
    print(f"Val distribution: {dict(zip(unique_val, counts_val))}")

    # Apply SMOTE to training fold
    X_tr_sm, y_tr_sm, sid_tr_sm, pfeat_tr_sm = apply_smote_to_fold(
        X_tr, y_tr, sid_tr, pfeat_tr,
        target_counts=SMOTE_TARGET_COUNTS,
        random_state=SEED + fold,
    )

    # Create datasets and loaders
    train_dataset = ArrayDataset(X_tr_sm, y_tr_sm, sid_tr_sm, pfeat_tr_sm)
    val_dataset = ArrayDataset(X_val, y_val, sid_val, pfeat_val)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # ADVICE 08/11: Compute class weights from SMOTE-resampled data
    unique_classes, class_counts_arr = np.unique(y_tr_sm, return_counts=True)
    class_counts = dict(zip(unique_classes, class_counts_arr))
    
    # Build weights array for all classes
    class_weights = np.ones(NUM_CLASSES, dtype=np.float32)
    for cls in range(NUM_CLASSES):
        count = class_counts.get(cls, 1)
        class_weights[cls] = 1.0 / (count + 1e-6)
    
    class_weights = class_weights / class_weights.mean()
    alpha = torch.tensor(class_weights, dtype=torch.float32).to(device)

    print(f"\nClass weights (after SMOTE): {class_weights}")

    # Initialize base model
    model = CNNBiLSTMAttention(
        input_size=INPUT_FEATURES,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        num_classes=NUM_CLASSES,
        dropout_rate=DROPOUT_RATE,
        patient_emb_dim=PATIENT_CLASS_EMB_DIM,
        num_patient_classes=3,
        cnn_channels=CNN_CHANNELS,
        cnn_kernel_size=5,
        num_attention_heads=NUM_ATTENTION_HEADS,
    ).to(device)

    # Loss function with focal loss and label smoothing
    criterion = FocalLossWithLabelSmoothing(
        gamma=FOCAL_GAMMA,
        alpha=alpha,
        smoothing=LABEL_SMOOTHING,
    )

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    # Training loop with early stopping
    best_val_f1 = 0.0
    best_epoch = 0
    patience_counter = 0

    print(f"\nStarting training for fold {fold}...")

    # Epoch loop
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()

        # Train
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, device, clip_grad_norm=1.0
        )

        # Validate
        val_loss, val_f1, eval_dict = validate_one_epoch(
            model, val_loader, criterion, device, int_to_label, NUM_CLASSES
        )

        epoch_time = time.time() - epoch_start

        if VERBOSE > 0 and epoch % VERBOSE == 0:
            print(f"Epoch {epoch:3d}/{EPOCHS} | "
                  f"Train: Loss={train_loss:.4f}, F1={train_f1:.4f} | "
                  f"Val: Loss={val_loss:.4f}, F1={val_f1:.4f} | "
                  f"Time: {epoch_time:.2f}s")

        # Early stopping check
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            patience_counter = 0

            # Save best model for this fold
            torch.save(model.state_dict(), f'models/fold_{fold}_best_model.pt')
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

    # Load best model
    model.load_state_dict(torch.load(f'models/fold_{fold}_best_model.pt'))

    # Final evaluation on validation set
    val_loss, val_f1, eval_dict = validate_one_epoch(
        model, val_loader, criterion, device, int_to_label, NUM_CLASSES
    )

    # Compute fold training time
    fold_time = time.time() - fold_start_time

    print(f"\n{'='*80}")
    print(f"FOLD {fold} RESULTS")
    print(f"{'='*80}")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation F1 (macro): {best_val_f1:.4f}")
    print(f"Final validation F1 (macro): {val_f1:.4f}")
    print(f"\nClassification Report:\n{eval_dict['report']}")
    print(f"\nConfusion Matrix:\n{eval_dict['confusion_matrix']}")
    print(f"\nFold training time: {fold_time:.2f}s")

    fold_results.append({
        'fold': fold,
        'best_epoch': best_epoch,
        'best_val_f1': best_val_f1,
        'final_val_f1': val_f1,
        'eval_dict': eval_dict,
    })

# Summary across folds
print(f"\n\n{'='*80}")
print("K-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*80}")

avg_f1 = np.mean([r['final_val_f1'] for r in fold_results])
std_f1 = np.std([r['final_val_f1'] for r in fold_results])

print(f"\nAverage validation F1 (macro): {avg_f1:.4f} ± {std_f1:.4f}")
print(f"\nPer-fold results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: F1={r['final_val_f1']:.4f} (best epoch: {r['best_epoch']})")

print(f"\n{'='*80}\n")

## 🎯 **Final Evaluation on Test Set**

In [ ]:
print(f"\n{'='*80}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*80}\n")

# Use the best fold model or ensemble
best_fold = max(fold_results, key=lambda x: x['final_val_f1'])['fold']
print(f"Using best fold: {best_fold}")

# Load best model from best fold
final_model = CNNBiLSTMAttention(
    input_size=INPUT_FEATURES,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout_rate=DROPOUT_RATE,
    patient_emb_dim=PATIENT_CLASS_EMB_DIM,
    num_patient_classes=3,
    cnn_channels=CNN_CHANNELS,
    cnn_kernel_size=5,
    num_attention_heads=NUM_ATTENTION_HEADS,
).to(device)

final_model.load_state_dict(torch.load(f'models/fold_{best_fold}_best_model.pt'))

# Create test loader
test_dataset = ArrayDataset(X_test, y_test, sid_test, pfeat_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Evaluate on test set
test_criterion = FocalLossWithLabelSmoothing(gamma=FOCAL_GAMMA, smoothing=0.0)  # No smoothing for evaluation

test_loss, test_f1, test_eval_dict = validate_one_epoch(
    final_model, test_loader, test_criterion, device, int_to_label, NUM_CLASSES
)

print(f"\nTest Set Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 (macro): {test_f1:.4f}")
print(f"\nClassification Report:\n{test_eval_dict['report']}")
print(f"\nConfusion Matrix:\n{test_eval_dict['confusion_matrix']}")

# Per-class metrics
test_labels = test_eval_dict['labels']
test_preds = test_eval_dict['preds']

# Per-class accuracy printout
for cls_idx in range(NUM_CLASSES):
    cls_mask = (test_labels == cls_idx)
    if cls_mask.sum() > 0:
        cls_acc = accuracy_score(test_labels[cls_mask], test_preds[cls_mask])
        print(f"\nClass {int_to_label[cls_idx]}:")
        print(f"  Accuracy: {cls_acc:.4f}")
        print(f"  Support: {cls_mask.sum()}")

print(f"\n{'='*80}\n")

## 📤 **Generate Submission CSV**

In [ ]:
print(f"\n{'='*80}")
print("GENERATING SUBMISSION CSV")
print(f"{'='*80}\n")

# Create submission loader
submission_dataset = ArrayDataset(X_submission, y_submission, sid_submission, pfeat_submission)
submission_loader = DataLoader(submission_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Get predictions for submission
_, _, _, submission_preds, submission_sids = validate_one_epoch(
    final_model, 
    submission_loader, 
    test_criterion, 
    device, 
    int_to_label, 
    NUM_CLASSES,
    return_predictions=True
)

# Convert predictions to labels
label_to_str = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}
submission_labels = [label_to_str[pred] for pred in submission_preds]

# Create submission dataframe
submission_df = pd.DataFrame({
    'sample_index': submission_sids,
    'label': submission_labels
})

# Sort by sample_index
submission_df = submission_df.sort_values('sample_index').reset_index(drop=True)

print(f"Submission shape: {submission_df.shape}")
print(f"\nSubmission label distribution:")
print(submission_df['label'].value_counts())
print(f"\nFirst few rows:")
print(submission_df.head(10))

# Save submission file
submission_filename = f'submission_{now}.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"\n✅ Submission saved to: {submission_filename}")

print(f"\n{'='*80}\n")

## 💾 **Save Final Model and Results**

In [ ]:
# Save final model
torch.save(final_model.state_dict(), f'models/final_model_{now}.pt')

# Save results summary
results_summary = {
    'timestamp': now,
    'hyperparameters': {
        'window_size': WINDOW_SIZE,
        'stride': STRIDE,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'dropout_rate': DROPOUT_RATE,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'focal_gamma': FOCAL_GAMMA,
        'label_smoothing': LABEL_SMOOTHING,
        'epochs': EPOCHS,
        'patience': PATIENCE,
    },
    'cv_results': {
        'avg_f1': float(avg_f1),
        'std_f1': float(std_f1),
        'fold_results': [
            {
                'fold': r['fold'],
                'best_epoch': r['best_epoch'],
                'final_val_f1': float(r['final_val_f1']),
            }
            for r in fold_results
        ],
    },
    'test_results': {
        'test_f1': float(test_f1),
        'test_loss': float(test_loss),
    },
    'submission_info': {
        'filename': submission_filename,
        'num_samples': int(len(submission_df)),
        'label_distribution': submission_df['label'].value_counts().to_dict(),
    },
}

import json
with open(f'models/results_summary_{now}.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nModel and results saved to models/ directory")
print(f"Final test F1 score: {test_f1:.4f}")
print(f"Submission file: {submission_filename}")
print(f"\n🎉 Training complete! 🎉")

print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"✅ Cross-validation F1: {avg_f1:.4f} ± {std_f1:.4f}")
print(f"✅ Test set F1: {test_f1:.4f}")
print(f"✅ Submission generated: {len(submission_df)} samples")
print(f"{'='*80}\n")